<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/6_3_Gaussian_Processes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VI. Kernel Methods

## Kernel Trick, Mercer's Theorem, and Kernel Methods

## Kernel PCA and Kernel Logistic Regression

## Gaussian Processes

### Тема 3. Гауссовские процессы

#### 1. Определение гауссовского процесса: ковариационная функция и совместная гауссовость

Ядровые методы, которые мы изучали до сих пор — SVM, KRR, KLR — решают задачи обучения путём поиска оптимальной функции в воспроизводящем ядерном гильбертовом пространстве (RKHS). Этот подход основан на минимизации регуляризованного эмпирического риска и приводит к точечным оценкам предсказаний. Однако во многих приложениях — от медицинской диагностики до робототехники — не менее важна, чем точечная оценка, **мера неопределённости** предсказания. Гауссовские процессы (Gaussian Processes, GP) предлагают вероятностную, байесовскую альтернативу: они задают априорное распределение непосредственно на пространстве функций и обновляют его с учётом наблюдений, получая апостериорное распределение с полноценной ковариационной структурой. В этом смысле GP являются естественным байесовским расширением ядровых методов, сохраняя тесную связь с ними через язык ковариационных функций (ядер).

---

##### 1.1. Определение гауссовского процесса

**Определение 1 (Гауссовский процесс).** *Гауссовским процессом* называется случайный процесс $\{f(\mathbf{x}) : \mathbf{x} \in \mathcal{X}\}$, такой что для любого конечного набора точек $\mathbf{x}_1, \dots, \mathbf{x}_n \in \mathcal{X}$ вектор значений $(f(\mathbf{x}_1), \dots, f(\mathbf{x}_n))^\top$ имеет многомерное нормальное распределение.

Гауссовский процесс полностью определяется своей **средней функцией** $m : \mathcal{X} \to \mathbb{R}$ и **ковариационной функцией** $k : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$:

$$
m(\mathbf{x}) = \mathbb{E}[f(\mathbf{x})], \qquad
k(\mathbf{x}, \mathbf{x}') = \operatorname{Cov}(f(\mathbf{x}), f(\mathbf{x}')) = \mathbb{E}\bigl[(f(\mathbf{x}) - m(\mathbf{x}))(f(\mathbf{x}') - m(\mathbf{x}'))\bigr].
$$

Это записывают как

$$
f(\mathbf{x}) \sim \mathcal{GP}\bigl(m(\mathbf{x}),\, k(\mathbf{x}, \mathbf{x}')\bigr).
$$

Из определения немедленно следует, что для любого конечного набора $\{\mathbf{x}_i\}_{i=1}^n$

$$
\begin{pmatrix} f(\mathbf{x}_1) \\ \vdots \\ f(\mathbf{x}_n) \end{pmatrix}
\sim \mathcal{N}\!\left(
\begin{pmatrix} m(\mathbf{x}_1) \\ \vdots \\ m(\mathbf{x}_n) \end{pmatrix},
\begin{pmatrix}
k(\mathbf{x}_1, \mathbf{x}_1) & \dots & k(\mathbf{x}_1, \mathbf{x}_n) \\
\vdots & \ddots & \vdots \\
k(\mathbf{x}_n, \mathbf{x}_1) & \dots & k(\mathbf{x}_n, \mathbf{x}_n)
\end{pmatrix}
\right).
$$

Таким образом, гауссовский процесс — это естественное обобщение многомерного нормального распределения на бесконечномерный случай: вместо конечного вектора средних и ковариационной матрицы мы задаём функции. Ковариационная матрица конкретной выборки — это в точности ядерная матрица $\mathbf{K}$ с элементами $\mathbf{K}_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$. Следовательно, ковариационная функция $k$ должна быть положительно определённым ядром в смысле определения раздела 2. Тем самым весь арсенал ядер, изученных нами ранее, — RBF, Matérn, полиномиальные, — немедленно переносится в мир гауссовских процессов.

На практике среднюю функцию часто полагают нулевой: $m(\mathbf{x}) \equiv 0$. Это не ограничивает общности, так как всегда можно вычесть из данных оценку среднего; кроме того, апостериорное среднее GP (как мы увидим) адаптируется к наблюдениям. Вся структура процесса — гладкость, характерный масштаб вариации, периодичность — кодируется ковариационной функцией.

---

##### 1.2. Связь с RKHS и ядровыми методами

Гауссовские процессы имеют глубокую связь с воспроизводящими ядерными гильбертовыми пространствами. Если $k$ — положительно определённое ядро, то оно порождает RKHS $\mathcal{H}_k$, состоящее из функций, норма которых контролирует гладкость. С другой стороны, $k$ определяет априорное распределение GP. Связь между ними двусторонняя:

- *От RKHS к GP*: для любого $f \in \mathcal{H}_k$ с нормой $\|f\|_{\mathcal{H}_k}$ можно построить вероятностную меру, придающую больший вес функциям с малой нормой. Фактически, априорное распределение GP с ядром $k$ является «гауссовской мерой», носитель которой лежит в $\mathcal{H}_k$ (точнее, в его пополнении по норме, немного более слабой, чем RKHS-норма).
- *Байесовский вывод в GP*: как мы видели в разделе 5.6, апостериорное среднее GP для регрессии с гауссовским шумом в точности совпадает с предсказанием ядерной гребневой регрессии (KRR) при $\lambda = \sigma^2_{\text{noise}}$. Однако GP идёт дальше, предоставляя полную апостериорную ковариацию, которая количественно оценивает неопределённость предсказания.

Таким образом, GP — это байесовский аналог ядровых методов. Вместо точечной оптимизации функции в RKHS мы формулируем вероятностное распределение над всеми функциями, совместимыми с ядром, и обновляем его согласно теореме Байеса. Эта дуальность объясняет, почему GP наследует многие свойства ядровых методов (например, теорему о представителе в виде того, что апостериорное среднее лежит в линейной оболочке $k(\cdot, \mathbf{x}_i)$), но в то же время предлагает richer inference — доверительные интервалы, сэмплирование функций, маргинализацию гиперпараметров.

---

##### 1.3. Примеры ковариационных функций

Выбор ковариационной функции — центральный шаг при построении GP. Она выражает наши априорные предположения о свойствах моделируемой функции: гладкости, характерном масштабе изменений, периодичности. Приведём наиболее употребительные семейства, предполагая стационарность ($k(\mathbf{x}, \mathbf{x}') = k(\mathbf{x} - \mathbf{x}')$) и изотропность ($k(\mathbf{x}, \mathbf{x}') = k(r)$, $r = \|\mathbf{x} - \mathbf{x}'\|$).

1. **Квадратичная экспонента (RBF, Squared Exponential):**
   $$
   k_{\text{SE}}(r) = \sigma_f^2 \exp\!\Bigl(-\frac{r^2}{2\ell^2}\Bigr).
   $$
   Параметр $\sigma_f^2$ — сигнальная дисперсия (масштаб функции по вертикали), $\ell$ — характерная длина (horizontal scale). Это ядро порождает **бесконечно дифференцируемые** (аналитические) траектории. Оно является самым популярным выбором для моделирования гладких функций.

2. **Ядро Матерна (Matérn):**
   $$
   k_{\text{Mat}}(r) = \sigma_f^2 \frac{2^{1-\nu}}{\Gamma(\nu)} \left(\frac{\sqrt{2\nu}\,r}{\ell}\right)^{\nu} K_{\nu}\!\left(\frac{\sqrt{2\nu}\,r}{\ell}\right),
   $$
   где $K_{\nu}$ — модифицированная функция Бесселя второго рода, $\nu > 0$ — параметр гладкости. При $\nu = 1/2$ получаем ядро Орнштейна–Уленбека (экспоненциальное), порождающее непрерывные, но **не дифференцируемые** траектории. При $\nu = 3/2$ траектории один раз дифференцируемы; при $\nu = 5/2$ — дважды. В пределе $\nu \to \infty$ ядро Матерна сходится к квадратичной экспоненте. Гибкость в выборе $\nu$ делает Matérn привлекательным, когда априори известна (или предполагается) конечная гладкость.

3. **Рациональное квадратичное (Rational Quadratic):**
   $$
   k_{\text{RQ}}(r) = \sigma_f^2 \left(1 + \frac{r^2}{2\alpha\ell^2}\right)^{-\alpha}, \quad \alpha > 0.
   $$
   Может быть представлено как смесь RBF-ядер с различными длинами $\ell$, распределёнными по гамма-распределению. Параметр $\alpha$ управляет «тяжестью хвостов»: при $\alpha \to \infty$ RQ переходит в RBF. Это ядро удобно для моделирования процессов, варьирующихся на нескольких масштабах.

4. **Периодическое ядро:**
   $$
   k_{\text{Per}}(r) = \sigma_f^2 \exp\!\Bigl(-\frac{2\sin^2(\pi r / p)}{\ell^2}\Bigr),
   $$
   где $p$ — период. Оно порождает периодические функции с периодом $p$ и используется, например, для моделирования сезонных эффектов.

Все перечисленные ядра являются положительно определёнными и, следовательно, законными ковариационными функциями.

---

##### 1.4. Свойства ковариационных функций и их влияние на траектории

Ковариационная функция полностью определяет свойства выборочных траекторий GP. Ключевые характеристики:

- **Стационарность.** Ядро зависит только от разности аргументов: $k(\mathbf{x}, \mathbf{x}') = k(\mathbf{x} - \mathbf{x}')$. Это означает, что статистические свойства процесса инвариантны относительно сдвигов. Все приведённые выше ядра (RBF, Matérn, RQ) стационарны.
- **Изотропность.** Ядро зависит только от расстояния $r = \|\mathbf{x} - \mathbf{x}'\|$, но не от направления. Изотропные ядра — частный случай стационарных.
- **Гладкость.** Гладкость выборочных траекторий напрямую связана с гладкостью ядра в нуле. А именно, если $k(r)$ имеет $2p$ производных в нуле, то траектории GP $p$ раз дифференцируемы в среднем квадратичном. RBF-ядро аналитично в нуле, отсюда бесконечная гладкость. Экспоненциальное ядро (Matérn $\nu=1/2$) имеет излом при $r=0$ ($k(r) = \sigma_f^2 e^{-r/\ell}$), и его траектории непрерывны, но не дифференцируемы (броуновское движение). Ядро Matérn $\nu=3/2$ даёт один раз дифференцируемые функции, и т.д.
- **Положительная определённость.** Ковариационная функция обязана быть ПО ядром, что гарантирует неотрицательную определённость любой матрицы $\mathbf{K}$ и, следовательно, корректность многомерного нормального распределения.

Выбор ядра — это способ ввести в модель априорное знание о структуре функции. Например, для моделирования данных с выраженной гладкостью выбирают RBF или Matérn с большим $\nu$; для данных с резкими перепадами — Matérn с $\nu=1/2$ или $\nu=3/2$; для сезонных данных добавляют периодическую компоненту, умножая или складывая ядра (поскольку сумма и произведение ПО ядер — ПО ядро, свойства замкнутости из раздела 4.1 сохраняются и здесь).

---

##### 1.5. Углублённый взгляд: спектральное представление стационарного GP

> **Для читателя с математической подготовкой.** Теорема Бохнера (раздел 2.5) устанавливает, что любое непрерывное стационарное положительно определённое ядро $k(\boldsymbol{\tau})$, $\boldsymbol{\tau} = \mathbf{x} - \mathbf{x}'$, представимо как преобразование Фурье конечной неотрицательной меры $S(\boldsymbol{\omega})$:
> $$
> k(\boldsymbol{\tau}) = \int_{\mathbb{R}^D} e^{2\pi i \boldsymbol{\tau}^\top \boldsymbol{\omega}} \, dS(\boldsymbol{\omega}).
> $$
> Функция $S$ называется **спектральной плотностью** (если мера абсолютно непрерывна) или спектральной мерой. Для RBF-ядра $k(\boldsymbol{\tau}) = \sigma_f^2 \exp(-\|\boldsymbol{\tau}\|^2 / (2\ell^2))$ спектральная плотность также является гауссовской:
> $$
> S(\boldsymbol{\omega}) = \sigma_f^2 (2\pi\ell^2)^{D/2} \exp(-2\pi^2 \ell^2 \|\boldsymbol{\omega}\|^2).
> $$
>
> Это представление имеет глубокие следствия. Во-первых, оно позволяет генерировать выборочные траектории GP с заданным ядром через обратное преобразование Фурье: можно сэмплировать коэффициенты Фурье с дисперсиями, пропорциональными $S(\boldsymbol{\omega})$, и восстанавливать функцию. Этот метод лежит в основе **Random Fourier Features** для масштабирования ядровых методов.
>
> Во-вторых, спектральный анализ объясняет связь гладкости ядра и убывания спектральной плотности. Чем быстрее убывает $S(\boldsymbol{\omega})$ с ростом $\|\boldsymbol{\omega}\|$, тем меньше в процессе высокочастотных компонент, и тем глаже траектории. Экспоненциальное убывание (как у RBF) даёт бесконечную гладкость; степенное убывание $S(\boldsymbol{\omega}) \sim \|\boldsymbol{\omega}\|^{-(2\nu + D)}$ соответствует ядру Матерна с параметром $\nu$ и порождает $\nu$ раз дифференцируемые функции.
>
> Таким образом, выбор ковариационной функции эквивалентен заданию спектрального состава процесса, и теорема Бохнера предоставляет удобный язык для описания этих свойств в частотной области.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое гауссовский процесс? Чем он отличается от обычного многомерного нормального распределения?
2. Какую роль играет ковариационная функция в GP? Приведите два примера ковариационных функций и опишите, какие функции они порождают.
3. Почему ковариационная функция должна быть положительно определённым ядром? Как это связано с ядерными методами?
4. Зачем нужна средняя функция $m(\mathbf{x})$? Можно ли её всегда полагать нулевой?
5. Что означает стационарность и изотропность ковариационной функции? Приведите пример неизотропного ядра.

**Для профессионалов:**

1. Докажите, что GP с нулевым средним и RBF-ядром порождает бесконечно дифференцируемые (аналитические) траектории. (Указание: используйте свойства производных ковариационной функции и критерий дифференцируемости в среднем квадратичном.)
2. Приведите пример ядра, дающего непрерывные, но нигде не дифференцируемые траектории. Какой параметр $\nu$ ядра Матерна этому соответствует?
3. Объясните, как построить периодический GP: запишите ковариационную функцию, порождающую строго периодические выборочные функции. (Указание: можно использовать ядро $k(x, x') = \sigma_f^2 \exp(-2\sin^2(\pi(x-x')/p)/\ell^2)$.)
4. Выведите выражение для условного распределения GP: дана обучающая выборка $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, где $y_i = f(\mathbf{x}_i) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0,\sigma_n^2)$. Найдите $p(f_* \mid \mathbf{X}, \mathbf{y}, \mathbf{X}_*)$.
5. Покажите, что байесовская линейная регрессия с гауссовским априором на веса $\mathbf{w} \sim \mathcal{N}(0, \Sigma)$ эквивалентна GP с ядром $k(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^\top \Sigma \phi(\mathbf{x}')$, где $\phi$ — явное отображение в признаковое пространство. Как это связано с RKHS?

### 2. Предсказательное распределение в гауссовской процессной регрессии

Определив гауссовский процесс как вероятностное распределение над функциями, мы естественно приходим к задаче регрессии: наблюдая зашумлённые значения скрытой функции в конечном наборе точек, мы хотим предсказать её поведение в новых точках, а также количественно оценить неопределённость этих предсказаний. Байесовская природа GP позволяет сделать это аналитически — благодаря свойству совместной гауссовости все маргинальные и условные распределения также гауссовские и выражаются в замкнутой форме через операции линейной алгебры. В этом разделе мы выведем фундаментальные уравнения предсказания для гауссовской процессной регрессии.

---

#### 2.1. Модель наблюдений

Рассмотрим стандартную модель регрессии с аддитивным гауссовским шумом. Пусть скрытая функция $f(\mathbf{x})$ априори является гауссовским процессом с нулевым средним и ковариационной функцией $k$:

$$
f(\mathbf{x}) \sim \mathcal{GP}\bigl(0, k(\mathbf{x}, \mathbf{x}')\bigr).
$$

Мы наблюдаем значения $y_i$ в точках $\mathbf{x}_i$, $i = 1,\dots,n$, зашумлённые независимым гауссовским шумом с дисперсией $\sigma_n^2$:

$$
y_i = f(\mathbf{x}_i) + \varepsilon_i, \qquad \varepsilon_i \overset{\text{i.i.d.}}{\sim} \mathcal{N}(0, \sigma_n^2).
$$

Шум $\varepsilon_i$ не зависит от $f$. Следовательно, совместное распределение наблюдений $\mathbf{y} = (y_1,\dots,y_n)^\top$ при фиксированном наборе входов $\mathbf{X} = \{\mathbf{x}_i\}_{i=1}^n$ является гауссовским:

$$
\mathbf{y} \mid \mathbf{X} \sim \mathcal{N}(\mathbf{0}, \mathbf{K} + \sigma_n^2 \mathbf{I}),
$$

где $\mathbf{K} \in \mathbb{R}^{n \times n}$ — матрица Грама с элементами $\mathbf{K}_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$, а $\mathbf{I}$ — единичная матрица. Добавление $\sigma_n^2 \mathbf{I}$ отражает вклад независимого шума: $\operatorname{Cov}(y_i, y_j) = k(\mathbf{x}_i, \mathbf{x}_j) + \sigma_n^2 \delta_{ij}$.

---

#### 2.2. Совместное распределение обучающих и тестовых точек

Пусть мы хотим сделать предсказание в $m$ тестовых точках $\mathbf{X}_* = \{\mathbf{x}_{*1}, \dots, \mathbf{x}_{*m}\}$. Обозначим соответствующие значения скрытой функции как $\mathbf{f}_* = (f(\mathbf{x}_{*1}), \dots, f(\mathbf{x}_{*m}))^\top$. По определению GP, совместное распределение $\mathbf{f}$ (значения на обучающих точках) и $\mathbf{f}_*$ является многомерным нормальным. С учётом шума $\boldsymbol{\varepsilon}$ для обучающих наблюдений $\mathbf{y} = \mathbf{f} + \boldsymbol{\varepsilon}$, совместная ковариационная структура имеет вид

$$
\begin{bmatrix} \mathbf{y} \\ \mathbf{f}_* \end{bmatrix}
\sim \mathcal{N}\!\left(
\mathbf{0},
\begin{bmatrix}
\mathbf{K} + \sigma_n^2 \mathbf{I} & \mathbf{K}_* \\
\mathbf{K}_*^\top & \mathbf{K}_{**}
\end{bmatrix}
\right). \tag{2.1}
$$

Здесь введены следующие матрицы:

- $\mathbf{K} = k(\mathbf{X}, \mathbf{X}) \in \mathbb{R}^{n \times n}$ — ядерная матрица между обучающими точками,
- $\mathbf{K}_* = k(\mathbf{X}, \mathbf{X}_*) \in \mathbb{R}^{n \times m}$ — матрица перекрёстных ковариаций между обучающими и тестовыми точками, элементы $(\mathbf{K}_*)_{ij} = k(\mathbf{x}_i, \mathbf{x}_{*j})$,
- $\mathbf{K}_{**} = k(\mathbf{X}_*, \mathbf{X}_*) \in \mathbb{R}^{m \times m}$ — ядерная матрица между тестовыми точками.

Среднее всех компонент равно нулю в силу нулевой средней функции априорного GP. Блок $\mathbf{K} + \sigma_n^2 \mathbf{I}$ есть ковариационная матрица наблюдений $\mathbf{y}$, блок $\mathbf{K}_*$ — ковариация между $\mathbf{y}$ и $\mathbf{f}_*$ (поскольку шум $\boldsymbol{\varepsilon}$ некоррелирован с $f(\mathbf{x}_{*j})$, ковариация $\operatorname{Cov}(\varepsilon_i, f(\mathbf{x}_{*j})) = 0$), а блок $\mathbf{K}_{**}$ — априорная ковариация тестовых значений.

---

#### 2.3. Условное распределение (апостериор)

По свойству многомерного нормального распределения, условное распределение $\mathbf{f}_*$ при фиксированных $\mathbf{y}$ также является гауссовским. Используя стандартные формулы для условных распределений (или дополнение Шура), получаем

$$
\mathbf{f}_* \mid \mathbf{X}, \mathbf{y}, \mathbf{X}_* \sim \mathcal{N}\bigl(\bar{\mathbf{f}}_*, \operatorname{cov}(\mathbf{f}_*)\bigr),
$$

где

$$
\bar{\mathbf{f}}_* = \mathbf{K}_*^\top (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y}, \tag{2.2}
$$

$$
\operatorname{cov}(\mathbf{f}_*) = \mathbf{K}_{**} - \mathbf{K}_*^\top (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{K}_*. \tag{2.3}
$$

Вывод основан на том, что для совместного нормального вектора

$$
\begin{bmatrix} \mathbf{a} \\ \mathbf{b} \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} \boldsymbol{\mu}_a \\ \boldsymbol{\mu}_b \end{bmatrix},
\begin{bmatrix} \boldsymbol{\Sigma}_{aa} & \boldsymbol{\Sigma}_{ab} \\ \boldsymbol{\Sigma}_{ba} & \boldsymbol{\Sigma}_{bb} \end{bmatrix}
\right),
$$

условное распределение $\mathbf{b} \mid \mathbf{a}$ имеет среднее $\boldsymbol{\mu}_b + \boldsymbol{\Sigma}_{ba} \boldsymbol{\Sigma}_{aa}^{-1} (\mathbf{a} - \boldsymbol{\mu}_a)$ и ковариацию $\boldsymbol{\Sigma}_{bb} - \boldsymbol{\Sigma}_{ba} \boldsymbol{\Sigma}_{aa}^{-1} \boldsymbol{\Sigma}_{ab}$. Подставляя $\boldsymbol{\mu}_a = \mathbf{0}$, $\boldsymbol{\mu}_b = \mathbf{0}$, $\boldsymbol{\Sigma}_{aa} = \mathbf{K} + \sigma_n^2 \mathbf{I}$, $\boldsymbol{\Sigma}_{ab} = \mathbf{K}_*$, $\boldsymbol{\Sigma}_{ba} = \mathbf{K}_*^\top$, $\boldsymbol{\Sigma}_{bb} = \mathbf{K}_{**}$, немедленно получаем (2.2)–(2.3).

Формула (2.2) показывает, что апостериорное среднее является линейной комбинацией ядерных функций, центрированных на обучающих точках: $\bar{f}(\mathbf{x}_*) = \sum_{i=1}^n \alpha_i k(\mathbf{x}_*, \mathbf{x}_i)$, где $\boldsymbol{\alpha} = (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y}$. Это в точности совпадает с предсказанием ядерной гребневой регрессии (KRR) при $\lambda = \sigma_n^2$. Однако GP идёт дальше, предоставляя ковариационную матрицу (2.3), которая количественно оценивает апостериорную неопределённость предсказания.

---

#### 2.4. Предсказательное распределение для $y_*$

Часто нас интересует не только сама функция $f$, но и зашумлённое наблюдение $y_* = f(\mathbf{x}_*) + \varepsilon_*$ в новой точке. Поскольку шум $\varepsilon_* \sim \mathcal{N}(0, \sigma_n^2)$ независим от $f$ и от обучающих данных, дисперсия $y_*$ равна $\operatorname{var}(f(\mathbf{x}_*)) + \sigma_n^2$. Следовательно, предсказательное распределение для $y_*$ есть

$$
y_* \mid \mathbf{X}, \mathbf{y}, \mathbf{x}_* \sim \mathcal{N}\bigl(\bar{f}_*, \operatorname{var}(f(\mathbf{x}_*)) + \sigma_n^2\bigr). \tag{2.4}
$$

Для множественных тестовых точек аналогично: $\mathbf{y}_* \sim \mathcal{N}(\bar{\mathbf{f}}_*, \operatorname{cov}(\mathbf{f}_*) + \sigma_n^2 \mathbf{I})$.

Эта дисперсия имеет прозрачную структуру: $\operatorname{cov}(\mathbf{f}_*)$ отражает неуверенность в значении самой функции (эпистемическую неопределённость), а $\sigma_n^2 \mathbf{I}$ — неустранимый шум наблюдений (алеаторическую неопределённость). Первая убывает с ростом числа близких обучающих точек, вторая остаётся константной.

---

#### 2.5. Интерпретация: ядровая регрессия с байесовскими интервалами

Формула для среднего (2.2) идентична предсказанию KRR, где $\lambda$ заменяется на $\sigma_n^2$. Следовательно, апостериорное среднее GP минимизирует регуляризованный эмпирический риск с квадратичной потерей и RKHS-штрафом, а регуляризация соответствует априорному предположению о гладкости, выраженному через ядро, и уровню шума. Однако GP предоставляет гораздо более полную информацию:

- **Доверительные интервалы:** диагональные элементы $\operatorname{cov}(\mathbf{f}_*)$ дают байесовские доверительные интервалы для скрытой функции, которые можно визуализировать как «трубку» вокруг предсказанной кривой.
- **Сэмплирование:** из многомерного нормального распределения можно генерировать выборочные траектории, совместимые с наблюдениями и априорными предположениями.
- **Оптимизация гиперпараметров:** маргинальное правдоподобие данных $p(\mathbf{y} \mid \mathbf{X})$ позволяет выбирать параметры ядра (например, $\ell$, $\sigma_f^2$) и дисперсию шума $\sigma_n^2$, максимизируя правдоподобие (evidence maximization).

Именно эта комбинация аналитической разрешимости, вероятностной интерпретации и возможности обучения гиперпараметров делает гауссовские процессы одним из наиболее привлекательных инструментов вероятностного машинного обучения.

---

#### 2.6. Углублённый взгляд: связь с MAP-оценкой и логарифмическое правдоподобие

> **Для читателя с математической подготовкой.** Глубокое понимание GP-регрессии достигается через анализ двух ключевых объектов: апостериорной ковариации как дополнения Шура и логарифмического маргинального правдоподобия.
>
> **Дополнение Шура.** Выражение (2.3) для апостериорной ковариации $\operatorname{cov}(\mathbf{f}_*)$ есть в точности дополнение Шура блока $\mathbf{K} + \sigma_n^2 \mathbf{I}$ в общей ковариационной матрице (2.1). Это отражает фундаментальное свойство гауссовских распределений: после наблюдения $\mathbf{y}$ неопределённость в $\mathbf{f}_*$ уменьшается на величину, зависящую от того, насколько сильно $\mathbf{f}_*$ коррелирует с $\mathbf{y}$. Если тестовая точка далека от обучающих, элементы $\mathbf{K}_*$ малы, и апостериорная ковариация приближается к априорной $\mathbf{K}_{**}$. Если же тестовая точка близка к обучающим, апостериорная дисперсия может стать значительно меньше априорной.
>
> **MAP-оценка.** Максимизация апостериорной вероятности (MAP) для $\mathbf{f}$ при фиксированных гиперпараметрах эквивалентна минимизации регуляризованного квадратичного функционала: $-\log p(\mathbf{f} \mid \mathbf{X}, \mathbf{y}) \propto \frac{1}{2\sigma_n^2}\|\mathbf{y} - \mathbf{f}\|^2 + \frac{1}{2} \mathbf{f}^\top \mathbf{K}^{-1} \mathbf{f}$. Минимум достигается при $\hat{\mathbf{f}} = \mathbf{K}(\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y}$, что совпадает с KRR. Следовательно, KRR можно интерпретировать как MAP-решение в GP-модели.
>
> **Логарифмическое маргинальное правдоподобие.** Для выбора гиперпараметров $\boldsymbol{\theta}$ (параметров ядра и $\sigma_n^2$) максимизируют логарифмическое правдоподобие наблюдений:
> $$
> \log p(\mathbf{y} \mid \mathbf{X}, \boldsymbol{\theta}) = -\frac{1}{2} \mathbf{y}^\top (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y} - \frac{1}{2} \log \det(\mathbf{K} + \sigma_n^2 \mathbf{I}) - \frac{n}{2} \log 2\pi.
> $$
> Первое слагаемое измеряет качество подгонки данных, второе — штраф за сложность модели (через объём ковариационной матрицы). Эта процедура, известная как максимизация правдоподобия II типа (evidence approximation), автоматически реализует компромисс между смещением и дисперсией, предотвращая переобучение без необходимости внешней кросс-валидации (хотя на практике их часто комбинируют).
>
> **Связь с байесовской линейной регрессией.** Если ядро представимо в виде $k(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^\top \boldsymbol{\Sigma} \phi(\mathbf{x}')$ для некоторого (возможно, бесконечномерного) отображения $\phi$, то GP с этим ядром эквивалентен байесовской линейной регрессии с априорным распределением на веса $\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \boldsymbol{\Sigma})$ и признаками $\phi(\mathbf{x})$. Это устанавливает прямую связь между GP, RKHS и линейными моделями с гауссовскими априорами, которая лежит в основе многих методов масштабирования GP (например, через случайные признаки).

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Как записывается совместное распределение $\mathbf{y}$ (обучающих наблюдений) и $\mathbf{f}_*$ (значений функции в тестовых точках)? Что входит в ковариационную матрицу?
2. Что такое матрица $\mathbf{K}_*$ и какова её размерность? Приведите формулу для её элементов.
3. Как получить предсказание среднего значения функции в новой точке $\mathbf{x}_*$? Сравните полученную формулу с предсказанием KRR.
4. Зачем нужна дисперсия предсказания в GP? Как она помогает оценить надёжность прогноза?
5. Чем принципиально отличается GP от KRR при одинаковом предсказании среднего? Какие дополнительные возможности даёт GP?

**Для профессионалов:**

1. Выведите формулу для апостериорной ковариации $\operatorname{cov}(\mathbf{f}_*)$ через дополнение Шура и покажите, что она всегда не превосходит априорную ковариацию $\mathbf{K}_{**}$ в смысле матричной неопределённости.
2. Докажите, что MAP-оценка для $\mathbf{f}$ в GP-регрессии совпадает с решением ядерной гребневой регрессии при $\lambda = \sigma_n^2$. Какие априорные предположения соответствуют этому штрафу?
3. Как учитывать гетероскедастичность (зависимость дисперсии шума от $\mathbf{x}$) в GP? Предложите модель и запишите модифицированное логарифмическое правдоподобие.
4. Выведите выражение для логарифмического маргинального правдоподобия $\log p(\mathbf{y} \mid \mathbf{X})$ и объясните роль каждого слагаемого.
5. Сравните GP с байесовской линейной регрессией, использующей ядерное расширение признаков $\phi$. Покажите, что предсказательные распределения совпадают, если ядро имеет вид $k(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^\top \boldsymbol{\Sigma} \phi(\mathbf{x}')$. Как это связано с RKHS?

### 3. Реализация гауссовской процессной регрессии на Python и визуализация

Теоретические формулы апостериорного среднего и ковариации, выведенные в предыдущем разделе, допускают прямую и элегантную программную реализацию. В этом разделе мы опишем структуру класса `GaussianProcessRegressor`, реализующего обучение и предсказание «с нуля», проиллюстрируем его работу на простом регрессионном примере, исследуем влияние гиперпараметров ядра и сравним GP с ядерной гребневой регрессией (KRR), подчеркнув уникальные возможности GP по оценке неопределённости.

---

#### 3.1. Класс `GaussianProcessRegressor`: структура и алгоритмы

Класс инкапсулирует ковариационную функцию (ядро), дисперсию шума и обученные параметры. Основные атрибуты:

- `kernel` — функция, принимающая два набора точек и возвращающая матрицу ковариаций (например, RBF-ядро с параметрами $\sigma_f^2$ и $\ell$).
- `sigma_n` — стандартное отклонение шума наблюдений ($\sigma_n$).
- `X_train_`, `y_train_` — обучающие данные.
- `L_` — нижнетреугольная матрица разложения Холецкого матрицы $\mathbf{K} + \sigma_n^2 \mathbf{I}$.
- `alpha_` — вектор $\boldsymbol{\alpha} = (\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1} \mathbf{y}$, используемый для вычисления среднего предсказания.

**Метод `fit(X, y)`** (обучение):

1. Сохранить обучающие данные.
2. Вычислить ядерную матрицу $\mathbf{K} = \text{kernel}(\mathbf{X}, \mathbf{X})$ размера $n \times n$.
3. Добавить шум к диагонали: $\mathbf{K}_y = \mathbf{K} + \sigma_n^2 \mathbf{I}$. Эта матрица симметрична и положительно определена при $\sigma_n > 0$ (даже если $\mathbf{K}$ вырождена, добавленная диагональ обеспечивает обратимость).
4. Выполнить разложение Холецкого: $\mathbf{K}_y = \mathbf{L} \mathbf{L}^\top$, где $\mathbf{L}$ — нижнетреугольная матрица. Разложение Холецкого численно устойчиво и требует $O(n^3/3)$ операций, что вдвое быстрее прямого обращения.
5. Вычислить $\boldsymbol{\alpha}$ в два этапа через решение треугольных систем:
   - Решить $\mathbf{L} \mathbf{z} = \mathbf{y}$ (прямая подстановка),
   - Решить $\mathbf{L}^\top \boldsymbol{\alpha} = \mathbf{z}$ (обратная подстановка).
   Результат $\boldsymbol{\alpha} = (\mathbf{K}_y)^{-1} \mathbf{y}$ используется далее.
6. Сохранить $\mathbf{L}$ и $\boldsymbol{\alpha}$.

**Метод `predict(X_new, return_std=True)`** (предсказание):

1. Вычислить матрицу перекрёстных ковариаций $\mathbf{K}_* = \text{kernel}(\mathbf{X}_{\text{new}}, \mathbf{X}_{\text{train}})$ размера $m \times n$.
2. Вычислить апостериорное среднее: $\bar{\mathbf{f}}_* = \mathbf{K}_* \boldsymbol{\alpha}$.
3. Для вычисления апостериорной ковариации (или её диагонали) используется сохранённое разложение $\mathbf{L}$:
   - Решить систему $\mathbf{L} \mathbf{V} = \mathbf{K}_*^\top$ (матрица $\mathbf{V}$ имеет размер $n \times m$).
   - Тогда апостериорная ковариация: $\operatorname{cov}(\mathbf{f}_*) = \mathbf{K}_{**} - \mathbf{V}^\top \mathbf{V}$, где $\mathbf{K}_{**} = \text{kernel}(\mathbf{X}_{\text{new}}, \mathbf{X}_{\text{new}})$.
   - Для получения стандартного отклонения берутся корни из диагональных элементов: $\text{std} = \sqrt{\text{diag}(\operatorname{cov}(\mathbf{f}_*))}$.
4. Если требуется предсказание для наблюдений $y_*$, к дисперсии добавляется $\sigma_n^2$.
5. Вернуть среднее и (опционально) стандартное отклонение.

**Численная устойчивость:** на практике при плохо обусловленной $\mathbf{K}$ к диагонали добавляют небольшой «джиттер» (jitter) порядка $10^{-8} \dots 10^{-6}$ перед разложением Холецкого, чтобы гарантировать положительную определённость.

---

#### 3.2. Пример регрессии: восстановление синуса с доверительной полосой

Классический тест для GP — восстановление функции $y = \sin(x)$ по зашумлённым наблюдениям. Сгенерируем $n = 10$ точек на интервале $[-5, 5]$, добавим гауссовский шум $\varepsilon \sim \mathcal{N}(0, 0.1^2)$. Обучим GP с RBF-ядром, параметры которого ($\ell=1.0$, $\sigma_f^2=1.0$, $\sigma_n^2=0.1$) пока фиксированы.

После вызова `fit(X, y)` и `predict(X_test)` на плотной сетке мы получаем среднее предсказание $\bar{f}(x)$ и стандартное отклонение $\text{std}(x)$. Визуализация включает:

- Истинную функцию $\sin(x)$ (пунктир).
- Обучающие точки (кружки).
- Предсказанное среднее (сплошная линия).
- 95% доверительную полосу (заливка между $\bar{f} \pm 1.96 \cdot \text{std}$).

В областях, где есть обучающие точки, полоса узкая, отражая высокую уверенность модели. В промежутках между точками и, особенно, за пределами обучающего интервала (экстраполяция) полоса расширяется, сигнализируя о росте неопределённости.

---

#### 3.3. Влияние гиперпараметров $\ell$ и $\sigma_f^2$

Параметры RBF-ядра — характерная длина $\ell$ и сигнальная дисперсия $\sigma_f^2$ — критически влияют на поведение GP.

- **Длина $\ell$** определяет типичный масштаб изменений функции. При слишком малом $\ell$ (например, $\ell = 0.2$) ядро становится узким: функция может быстро осциллировать, подстраиваясь под каждую точку, доверительная полоса резко сужается вблизи наблюдений, но между ними среднее возвращается к нулю, и полоса расширяется. Это ведёт к переобучению. При слишком большом $\ell$ ($\ell = 5.0$) ядро широкое: модель чрезмерно гладкая, она не успевает следовать за изгибами синуса, и полоса остаётся широкой даже вблизи точек — недообучение.
- **Сигнальная дисперсия $\sigma_f^2$** масштабирует амплитуду функции. Малое $\sigma_f^2$ заставляет среднее приближаться к нулю, полоса становится узкой по вертикали; большое $\sigma_f^2$ даёт больший размах колебаний.

Подбор $\ell$ и $\sigma_f^2$ (а также $\sigma_n^2$) часто осуществляют, максимизируя логарифмическое маргинальное правдоподобие. В собственной реализации для этого можно применить градиентный оптимизатор (L-BFGS), минимизируя отрицательный лог-правдоподобие (формула приведена в разделе 2.6), и аналитически вычисляя градиенты по гиперпараметрам через следовые операции с $(\mathbf{K} + \sigma_n^2 \mathbf{I})^{-1}$.

---

#### 3.4. Сравнение с KRR: точечные предсказания против байесовской неопределённости

Полезно сравнить GP с ядерной гребневой регрессией на том же примере. При $\lambda = \sigma_n^2$ предсказания средних у KRR и GP идентичны. Различие становится очевидным, когда мы смотрим на информативность вывода:

- **KRR** выдаёт только точечную оценку $\hat{y}(\mathbf{x}_*)$. Пользователь не знает, насколько можно доверять этому предсказанию в конкретной точке.
- **GP** дополнительно возвращает дисперсию, которая естественным образом мала в густо населённых областях и велика в областях экстраполяции. Эта дисперсия является байесовской мерой эпистемической неопределённости и может использоваться, например, в активном обучении для выбора следующих точек измерений или в байесовской оптимизации для баланса exploration/exploitation.

Таким образом, GP — это не просто «ещё один регрессор», а вероятностный фреймворк, который даёт полную картину предсказательной неопределённости, не прибегая к бутстрапу или другим эвристикам.

---

#### 3.5. Пример экстраполяции: рост дисперсии вне обучающих данных

Одно из наиболее наглядных свойств GP — поведение при экстраполяции. Обучим модель на точках из интервала $[-4, 4]$, а предсказание выполним на расширенном интервале $[-6, 6]$. Визуализация покажет:

- Внутри $[-4, 4]$ среднее хорошо отслеживает синус, полоса узкая.
- При выходе за пределы $[-4, 4]$ среднее стремится к априорному нулю, а дисперсия быстро возрастает, возвращаясь к априорной $\sigma_f^2 + \sigma_n^2$ (для RBF-ядра $\mathbf{K}_{**} = \sigma_f^2$ в удалённых точках, плюс шум). Это отражает потерю информации: вдали от данных модель «забывает» наблюдения и возвращается к своим априорным представлениям.

Такое поведение чрезвычайно полезно на практике: оно честно сигнализирует о том, что в областях, не покрытых данными, прогнозам доверять не следует. KRR, напротив, молча выдал бы какие-то числа без указания на их ненадёжность.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Зачем в реализации GP используется разложение Холецкого, а не прямое обращение матрицы? Какие преимущества это даёт?
2. Как вычислить логарифмическое маргинальное правдоподобие, используя уже найденное разложение Холецкого?
3. Что такое параметр длины (length-scale) $\ell$ в RBF-ядре? Как он влияет на форму предсказаний GP?
4. Как правильно добавить шум наблюдений в ковариационную матрицу? Почему добавление $\sigma_n^2$ только к диагональным элементам соответствует независимому шуму?
5. Почему сложность обучения GP составляет $O(n^3)$? Какая операция является самой затратной?

**Для профессионалов:**

1. Реализуйте в методе `fit` добавление «джиттера» (jitter) к диагонали $\mathbf{K} + \sigma_n^2 \mathbf{I}$ для улучшения численной устойчивости. Как выбрать величину джиттера?
2. Напишите метод, который оптимизирует гиперпараметры ядра ($\ell$, $\sigma_f^2$, $\sigma_n^2$) путём минимизации отрицательного логарифмического маргинального правдоподобия с помощью градиентного спуска или L-BFGS. Какие градиенты гиперпараметров вам понадобятся?
3. Добавьте в класс поддержку нескольких ядер, комбинируемых через сумму или произведение. Как это реализовать наиболее элегантно?
4. Сравните скорость предсказания вашей реализации с библиотекой GPyTorch для $n=10^4$. Какие оптимизации использует GPyTorch (Blackbox Matrix-Matrix умножение, сопряжённые градиенты, etc.)?
5. Объясните, как модифицировать GP для многомерного выхода (multi-output GP). Какие подходы существуют: независимые выходы, линейная модель корегионализации (LMC), свёрточные процессы?

### 4. Выбор гиперпараметров: максимизация маргинального правдоподобия

Одно из наиболее привлекательных свойств гауссовских процессов — возможность настраивать гиперпараметры (параметры ковариационной функции $\ell$, $\sigma_f^2$ и дисперсию шума $\sigma_n^2$) непосредственно из данных, не прибегая к перекрёстной проверке. Это достигается путём максимизации **маргинального правдоподобия** (marginal likelihood, или *evidence*) — вероятности наблюдений при интегрировании по всем возможным значениям скрытой функции. Данный подход, известный как *evidence approximation* или максимизация правдоподобия II типа, автоматически реализует компромисс между качеством подгонки и сложностью модели и лежит в основе практического применения GP.

---

#### 4.1. Маргинальное правдоподобие (evidence)

Рассмотрим модель наблюдений $\mathbf{y} = \mathbf{f} + \boldsymbol{\varepsilon}$, где $\mathbf{f} \sim \mathcal{N}(\mathbf{0}, \mathbf{K})$, $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma_n^2 \mathbf{I})$. Интегрируя скрытые значения $\mathbf{f}$, получаем маргинальное распределение наблюдений:

$$
\mathbf{y} \mid \mathbf{X}, \boldsymbol{\theta} \sim \mathcal{N}(\mathbf{0}, \mathbf{K}_y), \qquad \mathbf{K}_y = \mathbf{K} + \sigma_n^2 \mathbf{I},
$$

где $\boldsymbol{\theta}$ — совокупность всех гиперпараметров (например, $\ell$, $\sigma_f^2$, $\sigma_n^2$). Логарифмическое маргинальное правдоподобие имеет вид

$$
\log p(\mathbf{y} \mid \mathbf{X}, \boldsymbol{\theta}) = -\frac{1}{2} \mathbf{y}^\top \mathbf{K}_y^{-1} \mathbf{y} - \frac{1}{2} \log \det(\mathbf{K}_y) - \frac{n}{2} \log 2\pi. \tag{4.1}
$$

Три слагаемых имеют ясную интерпретацию:

- **Слагаемое подгонки данных** $-\frac{1}{2} \mathbf{y}^\top \mathbf{K}_y^{-1} \mathbf{y}$ измеряет, насколько хорошо модель описывает наблюдения. Оно единственное зависит от $\mathbf{y}$.
- **Штраф за сложность** $-\frac{1}{2} \log \det(\mathbf{K}_y)$ действует как регуляризатор: объём ковариационной матрицы растёт с увеличением $\sigma_f^2$ или $\ell$, и это слагаемое штрафует избыточную сложность.
- **Нормировочная константа** $-\frac{n}{2} \log 2\pi$ не зависит от гиперпараметров и может быть опущена при оптимизации.

Максимизация (4.1) одновременно поощряет хорошее соответствие данным и препятствует переобучению, автоматически находя оптимальный баланс.

---

#### 4.2. Градиенты по гиперпараметрам

Для эффективной оптимизации с помощью градиентных методов (например, L-BFGS) необходимо уметь вычислять производные $\log p(\mathbf{y} \mid \boldsymbol{\theta})$ по каждому гиперпараметру $\theta_j$. Дифференцируя (4.1) и используя матричные тождества $\frac{\partial \mathbf{K}_y^{-1}}{\partial \theta_j} = -\mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j} \mathbf{K}_y^{-1}$ и $\frac{\partial \log \det(\mathbf{K}_y)}{\partial \theta_j} = \operatorname{tr}\bigl(\mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j}\bigr)$, получаем общее выражение:

$$
\frac{\partial}{\partial \theta_j} \log p(\mathbf{y} \mid \boldsymbol{\theta}) = \frac{1}{2} \mathbf{y}^\top \mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j} \mathbf{K}_y^{-1} \mathbf{y} - \frac{1}{2} \operatorname{tr}\!\left(\mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j}\right). \tag{4.2}
$$

Остаётся подставить конкретный вид $\frac{\partial \mathbf{K}_y}{\partial \theta_j}$. Для широко используемого RBF-ядра

$$
k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\!\Bigl(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\Bigr),
$$

и аддитивного шума $\sigma_n^2$, производные таковы:

- **По $\ell$:**
  $$
  \frac{\partial k(\mathbf{x}, \mathbf{x}')}{\partial \ell} = \sigma_f^2 \exp\!\Bigl(-\frac{r^2}{2\ell^2}\Bigr) \cdot \frac{r^2}{\ell^3} = k(\mathbf{x}, \mathbf{x}') \cdot \frac{r^2}{\ell^3},
  $$
  где $r = \|\mathbf{x} - \mathbf{x}'\|$.

- **По $\sigma_f^2$:**
  $$
  \frac{\partial k(\mathbf{x}, \mathbf{x}')}{\partial \sigma_f^2} = \exp\!\Bigl(-\frac{r^2}{2\ell^2}\Bigr) = \frac{1}{\sigma_f^2} k(\mathbf{x}, \mathbf{x}').
  $$

- **По $\sigma_n^2$:** $\frac{\partial \mathbf{K}_y}{\partial \sigma_n^2} = \mathbf{I}$, поэтому соответствующее слагаемое в градиенте упрощается до $\frac{1}{2} \mathbf{y}^\top \mathbf{K}_y^{-2} \mathbf{y} - \frac{1}{2} \operatorname{tr}(\mathbf{K}_y^{-1})$, либо можно использовать общую формулу (4.2) с $\frac{\partial \mathbf{K}_y}{\partial \sigma_n^2} = \mathbf{I}$.

Имея аналитические градиенты, можно применять эффективные алгоритмы оптимизации.

---

#### 4.3. Реализация оптимизации гиперпараметров

Добавление функциональности оптимизации в класс `GaussianProcessRegressor` (описанный в разделе 3.1) естественно осуществляется через метод `fit` с флагом `optimize=True`. Логика такова:

1. **Инициализация гиперпараметров.** Задать начальные значения $\boldsymbol{\theta}_0 = (\log \ell, \log \sigma_f, \log \sigma_n)$ (логарифмирование гарантирует положительность). Начальные приближения можно выбрать эвристически: $\ell$ — как характерный разброс входных данных, $\sigma_f$ — как стандартное отклонение $\mathbf{y}$, $\sigma_n$ — как малую долю от $\sigma_f$.

2. **Определение целевой функции.** Реализовать функцию `neg_log_marginal_likelihood(theta)`, которая:
   - Преобразует `theta` обратно в $\ell, \sigma_f, \sigma_n$ через экспоненту.
   - Строит $\mathbf{K}$ с помощью RBF-ядра, формирует $\mathbf{K}_y = \mathbf{K} + \sigma_n^2 \mathbf{I}$.
   - Выполняет разложение Холецкого $\mathbf{K}_y = \mathbf{L} \mathbf{L}^\top$ (с добавлением джиттера при необходимости).
   - Вычисляет логарифмическое правдоподобие по формуле (4.1), используя сохранённое разложение: $\log \det(\mathbf{K}_y) = 2 \sum_i \log \mathbf{L}_{ii}$, а $\mathbf{y}^\top \mathbf{K}_y^{-1} \mathbf{y} = \|\mathbf{L}^{-1} \mathbf{y}\|^2$ (решается через треугольную систему).
   - Возвращает отрицательное значение (для минимизации).

3. **Градиенты.** Если доступны аналитические градиенты (4.2), реализовать функцию `grad_neg_log_marginal_likelihood(theta)`. Она вычисляет матрицы $\frac{\partial \mathbf{K}_y}{\partial \theta_j}$, решает $\mathbf{L} \mathbf{a} = \mathbf{y}$, затем вычисляет $\mathbf{b} = \mathbf{L}^{-\top} \mathbf{a} = \mathbf{K}_y^{-1} \mathbf{y}$, и матрицу $\mathbf{K}_y^{-1}$ (или её действие). Градиент собирается как $\frac12 \mathbf{b}^\top \frac{\partial \mathbf{K}_y}{\partial \theta_j} \mathbf{b} - \frac12 \operatorname{tr}(\mathbf{K}_y^{-1} \frac{\partial \mathbf{K}_y}{\partial \theta_j})$. Для ускорения след оценивают с помощью стохастического оценивания (Hutchinson's trace estimator) либо используют автоматическое дифференцирование. На практике часто полагаются на численные градиенты через `scipy.optimize.approx_fprime`, но аналитические предпочтительнее по скорости.

4. **Оптимизация.** Вызвать `scipy.optimize.minimize` с методом `L-BFGS-B`, передавая начальное приближение, границы (например, $[-10, 10]$ для логарифмов) и функцию с градиентом. После оптимизации обновить внутренние параметры ядра и $\sigma_n$, заново вычислить $\mathbf{K}_y$, разложение Холецкого и вектор $\boldsymbol{\alpha}$ для последующего предсказания.

---

#### 4.4. Пример оптимизации на данных

Вернёмся к примеру с зашумлённым синусом из раздела 3.2. Предположим, мы начали с заведомо неподходящих гиперпараметров, например $\ell=0.5$, $\sigma_f=2.0$, $\sigma_n=0.5$. GP с такими параметрами выдает либо чрезмерно извилистую функцию (если $\ell$ мала), либо слишком широкую доверительную полосу. После выполнения метода `fit` с флагом `optimize=True` оптимизатор L-BFGS за несколько десятков итераций находит значения, близкие к истинным ($\ell \approx 1.5$, $\sigma_f \approx 1.0$, $\sigma_n \approx 0.1$).

Результат после оптимизации:
- Предсказанное среднее практически идеально ложится на истинный синус.
- Доверительная полоса становится адекватно узкой вблизи точек и расширяется на экстраполяции.
- Значение логарифмического маргинального правдоподобия возрастает, отражая улучшение модели.

Такой автоматический подбор устраняет необходимость в ручной настройке и позволяет GP адаптироваться к данным.

---

#### 4.5. Предостережения: локальные минимумы и многомодальность

Несмотря на элегантность, максимизация маргинального правдоподобия сопряжена с рядом практических трудностей:

- **Многомодальность.** Поверхность $\log p(\mathbf{y} \mid \boldsymbol{\theta})$ как функция гиперпараметров часто имеет несколько локальных максимумов, особенно при малых объёмах данных. Оптимизатор градиентного типа может застрять в неоптимальном режиме (например, очень малая $\ell$ при большом шуме, либо большая $\ell$ при нулевом шуме).
- **Чувствительность к начальным значениям.** Результат оптимизации зависит от стартовой точки. Рекомендуется запускать оптимизацию несколько раз из различных случайных начальных приближений и выбирать решение с наибольшим значением правдоподобия.
- **Переобучение гиперпараметров.** Маргинальное правдоподобие, как и любой критерий, основанный на обучающих данных, может переоценивать качество модели, особенно когда $n$ мало. В таких случаях полезно дополнительно оценивать качество на отложенной выборке (кросс-валидация) или использовать полностью байесовский подход с интегрированием по гиперпараметрам (например, через MCMC или вариационный вывод).
- **Плохая обусловленность.** При очень малых $\sigma_n$ или сильно коррелированных данных матрица $\mathbf{K}_y$ может стать вырожденной, что приводит к неустойчивости вычислений. Добавление джиттера и аккуратная параметризация (логарифмическая шкала) смягчают, но не устраняют полностью эту проблему.

На практике эти трудности преодолеваются комбинацией аккуратной инициализации, нескольких рестартов, использования ансамблей GP с разными гиперпараметрами и, в критических случаях, переходом к вариационным методам (GPflow, GPyTorch), которые естественным образом устойчивее и масштабируемее.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое маргинальное правдоподобие (marginal likelihood) в контексте GP? Почему его удобно использовать для выбора гиперпараметров?
2. Из каких трёх основных компонент состоит логарифмическое маргинальное правдоподобие и какую роль играет каждая?
3. Как выглядят градиенты логарифмического правдоподобия по гиперпараметрам? Почему полезно иметь аналитические выражения?
4. Почему для оптимизации гиперпараметров часто используют алгоритм L-BFGS, а не простой градиентный спуск?
5. Какие проблемы могут возникнуть при максимизации маргинального правдоподобия? Назовите две и предложите способы их смягчения.

**Для профессионалов:**

1. Выведите общее выражение (4.2) для градиента логарифмического маргинального правдоподобия GP по произвольному гиперпараметру. Подставьте производные для RBF-ядра и получите явные формулы для $\partial / \partial \ell$ и $\partial / \partial \sigma_f^2$.
2. Как избежать переобучения гиперпараметров при малых выборках? Сравните стратегии: кросс-валидация, интегрирование по гиперпараметрам (MCMC), вариационный байесовский вывод.
3. Объясните, как EM-алгоритм мог бы быть применён для оптимизации гиперпараметров GP. Почему прямой градиентный подход обычно предпочтительнее?
4. Реализуйте совместную оптимизацию нескольких ядер (например, RBF + периодическое) с использованием evidence. Какие модификации кода и градиентов для этого потребуются?
5. Как автоматическое дифференцирование (autograd) в библиотеках GPflow или GPyTorch упрощает реализацию обучения GP? Приведите пример вычисления градиента без аналитического вывода.

### 5. Гауссовские процессы для классификации: лапласовское приближение

До сих пор мы рассматривали гауссовские процессы в контексте регрессии, где наблюдения — это значения скрытой функции, искажённые аддитивным гауссовским шумом. В этом случае всё аналитически разрешимо: апостериорное распределение и предсказания остаются гауссовскими. Однако в задачах классификации связь между скрытой функцией и наблюдаемыми метками нелинейна, что нарушает гауссовость. В результате точный байесовский вывод становится невозможным, и мы вынуждены прибегать к приближениям. Наиболее классическим из них является **лапласовское приближение**, которое аппроксимирует апостериорное распределение гауссовским с центром в моде апостериора (MAP-оценке) и ковариацией, определяемой кривизной логарифма апостериора в этой точке. Этот метод не только даёт эффективную процедуру обучения, но и сохраняет тесную связь с ядерной логистической регрессией (KLR), позволяя взглянуть на KLR как на MAP-решение в байесовской GP-модели.

---

#### 5.1. Вероятностная модель GP-классификации

Рассмотрим задачу бинарной классификации с метками $y_i \in \{0,1\}$. Введём скрытую (латентную) функцию $f : \mathcal{X} \to \mathbb{R}$, априорное распределение которой — гауссовский процесс с нулевым средним и ковариационной функцией $k$:

$$
f(\mathbf{x}) \sim \mathcal{GP}\bigl(0, k(\mathbf{x}, \mathbf{x}')\bigr).
$$

Условная вероятность положительного класса задаётся сигмоидной функцией, «сжимающей» значения $f(\mathbf{x})$ в интервал $[0,1]$:

$$
p(y = 1 \mid f(\mathbf{x})) = \sigma(f(\mathbf{x})), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}.
$$

Наблюдения $\mathbf{y} = (y_1,\dots,y_n)^\top$ предполагаются условно независимыми при заданных $f_i = f(\mathbf{x}_i)$. Тогда правдоподобие есть

$$
p(\mathbf{y} \mid \mathbf{f}) = \prod_{i=1}^n \sigma(f_i)^{y_i} (1 - \sigma(f_i))^{1 - y_i},
$$

а совместное апостериорное распределение скрытых значений $\mathbf{f} = (f_1,\dots,f_n)^\top$ принимает вид

$$
p(\mathbf{f} \mid \mathbf{X}, \mathbf{y}) = \frac{1}{Z} \, p(\mathbf{f} \mid \mathbf{X}) \, p(\mathbf{y} \mid \mathbf{f}),
$$

где априорное распределение $\mathbf{f} \sim \mathcal{N}(\mathbf{0}, \mathbf{K})$ с матрицей $\mathbf{K}_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$. Из-за нелинейной сигмоиды апостериорное распределение (5.1) **не является гауссовским**, и его нормализующая константа $Z = p(\mathbf{y} \mid \mathbf{X})$ не вычисляется аналитически. Именно это обстоятельство заставляет искать приближённые методы.

---

#### 5.2. Лапласовское приближение: нахождение моды апостериора

Лапласовский метод строит гауссовскую аппроксимацию апостериора в окрестности его максимума — **MAP-оценки** (maximum a posteriori). Логарифм ненормированного апостериора (с точностью до аддитивной константы) есть

$$
\Psi(\mathbf{f}) = \log p(\mathbf{f} \mid \mathbf{y}) + \text{const} = -\frac{1}{2} \mathbf{f}^\top \mathbf{K}^{-1} \mathbf{f} + \sum_{i=1}^n \Bigl[ y_i \log \sigma(f_i) + (1-y_i) \log\bigl(1 - \sigma(f_i)\bigr) \Bigr]. \tag{5.2}
$$

Заметим, что это выражение есть в точности регуляризованный логарифм правдоподобия с квадратичным штрафом $\frac{1}{2}\mathbf{f}^\top \mathbf{K}^{-1}\mathbf{f}$, где роль регуляризации играет обратная ковариационная матрица априорного GP.

Для нахождения $\mathbf{f}_{\text{MAP}} = \arg\max_{\mathbf{f}} \Psi(\mathbf{f})$ приравняем градиент к нулю:

$$
\nabla_{\mathbf{f}} \Psi = -\mathbf{K}^{-1} \mathbf{f} + (\mathbf{y} - \boldsymbol{\sigma}(\mathbf{f})) = \mathbf{0},
$$

где $\boldsymbol{\sigma}(\mathbf{f})_i = \sigma(f_i)$. Отсюда получаем нелинейное уравнение

$$
\mathbf{f} = \mathbf{K} \bigl(\mathbf{y} - \boldsymbol{\sigma}(\mathbf{f})\bigr). \tag{5.3}
$$

Из него видно, что решение $\mathbf{f}_{\text{MAP}}$ лежит в образе матрицы $\mathbf{K}$, т.е. представимо в виде линейной комбинации ядерных функций — как и предписывает теорема о представителе.

Уравнение (5.3) решается итеративно методом Ньютона. Гессиан $\Psi$ равен

$$
\nabla^2_{\mathbf{f}} \Psi = -\mathbf{K}^{-1} - \mathbf{W},
$$

где $\mathbf{W} = \operatorname{diag}(w_1,\dots,w_n)$ — диагональная матрица с элементами

$$
w_i = \sigma(f_i)(1 - \sigma(f_i)) > 0.
$$

Матрица $-\nabla^2 \Psi = \mathbf{K}^{-1} + \mathbf{W}$ положительно определена, поэтому $\Psi$ строго вогнута и имеет единственный максимум. Ньютоновский шаг обновления $\mathbf{f}^{\text{new}} = \mathbf{f} - (\nabla^2 \Psi)^{-1} \nabla \Psi$ после алгебраических преобразований сводится к решению линейной системы

$$
(\mathbf{K}^{-1} + \mathbf{W}) \mathbf{f}^{\text{new}} = \mathbf{W} \mathbf{f} + (\mathbf{y} - \boldsymbol{\sigma}(\mathbf{f})). \tag{5.4}
$$

Итерации продолжаются до сходимости, давая $\mathbf{f}_{\text{MAP}}$.

---

#### 5.3. Гауссовская аппроксимация апостериора

Лапласовское приближение заменяет истинное апостериорное распределение гауссовским с центром в $\mathbf{f}_{\text{MAP}}$ и ковариацией, равной обратному гессиану со знаком минус, вычисленному в точке максимума:

$$
p(\mathbf{f} \mid \mathbf{X}, \mathbf{y}) \approx \mathcal{N}\bigl( \mathbf{f}_{\text{MAP}}, \; \mathbf{\Sigma}_{\text{post}} \bigr), \qquad \mathbf{\Sigma}_{\text{post}} = \bigl(\mathbf{K}^{-1} + \mathbf{W}\bigr)^{-1}, \tag{5.5}
$$

где $\mathbf{W}$ теперь берётся при $\mathbf{f} = \mathbf{f}_{\text{MAP}}$. Эта аппроксимация оправдана тем, что апостериор унимодален и с ростом числа данных становится всё более похожим на гауссовский (асимптотическая нормальность по Бернштейну–фон Мизесу). Матрица $\mathbf{\Sigma}_{\text{post}}$ отражает неопределённость в значениях скрытой функции в обучающих точках.

Полезно переписать ковариацию в форме, более удобной для предсказаний. Применяя матричную лемму об обращении (формулу Вудбери) к $(\mathbf{K}^{-1} + \mathbf{W})^{-1}$, получаем

$$
\mathbf{\Sigma}_{\text{post}} = \mathbf{K} - \mathbf{K} \bigl(\mathbf{W}^{-1} + \mathbf{K}\bigr)^{-1} \mathbf{K}. \tag{5.6}
$$

Это представление, хотя и содержит $\mathbf{W}^{-1}$, удобно тем, что матрица $\mathbf{W}^{-1}$ диагональна и легко вычисляется.

---

#### 5.4. Предсказание для новой точки

Имея гауссовское приближение апостериора $\mathbf{f}$, мы можем получить предсказательное распределение для латентной функции $f_* = f(\mathbf{x}_*)$ в тестовой точке. Так как совместное априорное распределение $\mathbf{f}$ и $f_*$ гауссово, условное распределение $p(f_* \mid \mathbf{f})$ также гауссово:

$$
f_* \mid \mathbf{f} \sim \mathcal{N}\bigl( \mathbf{k}_*^\top \mathbf{K}^{-1} \mathbf{f}, \; k_{**} - \mathbf{k}_*^\top \mathbf{K}^{-1} \mathbf{k}_* \bigr),
$$

где $\mathbf{k}_* = k(\mathbf{X}, \mathbf{x}_*)$, $k_{**} = k(\mathbf{x}_*, \mathbf{x}_*)$. Интегрируя по приближённому апостериору $\mathbf{f} \sim \mathcal{N}(\mathbf{f}_{\text{MAP}}, \mathbf{\Sigma}_{\text{post}})$, получаем (благодаря линейности по $\mathbf{f}$) гауссовское предсказательное распределение для $f_*$:

$$
f_* \mid \mathbf{X}, \mathbf{y}, \mathbf{x}_* \;\approx\; \mathcal{N}(\mu_*, \sigma_*^2),
$$

где

$$
\mu_* = \mathbf{k}_*^\top \mathbf{K}^{-1} \mathbf{f}_{\text{MAP}}, \tag{5.7}
$$

$$
\sigma_*^2 = k_{**} - \mathbf{k}_*^\top (\mathbf{K} + \mathbf{W}^{-1})^{-1} \mathbf{k}_*. \tag{5.8}
$$

Вывод (5.8) опирается на формулу (5.6) и тождество $\mathbf{K}^{-1} \mathbf{\Sigma}_{\text{post}} \mathbf{K}^{-1} = \mathbf{K}^{-1} - (\mathbf{K} + \mathbf{W}^{-1})^{-1}$, которое проверяется прямой подстановкой.

Наконец, чтобы получить вероятность положительного класса, необходимо проинтегрировать сигмоиду по предсказательному распределению $f_*$:

$$
p(y_* = 1 \mid \mathbf{X}, \mathbf{y}, \mathbf{x}_*) \approx \int \sigma(f_*) \, \mathcal{N}(f_* \mid \mu_*, \sigma_*^2) \, df_*. \tag{5.9}
$$

Этот одномерный интеграл не берётся аналитически, но легко вычисляется численно (например, квадратурой Гаусса–Эрмита). Кроме того, существует удобная аппроксимация через функцию probit: $\int \sigma(f) \mathcal{N}(f \mid \mu, \sigma^2) df \approx \sigma\!\bigl(\mu / \sqrt{1 + \pi \sigma^2 / 8}\bigr)$. Такая аппроксимация даёт хорошую точность и широко применяется на практике.

Таким образом, GP-классификация с лапласовским приближением выдаёт не только наиболее вероятный класс, но и вероятность, учитывающую апостериорную неопределённость.

---

#### 5.5. Углублённый взгляд: связь с ядерной логистической регрессией (KLR)

> **Для читателя с математической подготовкой.** Лапласовское приближение для GP-классификации вскрывает глубокую связь с ядерной логистической регрессией (KLR). Вспомним задачу KLR (раздел 4.1): минимизировать функционал
> $$ \hat{J}(\boldsymbol{\alpha}) = -\sum_{i=1}^n \log p(y_i \mid f_i) + \frac{\lambda}{2} \boldsymbol{\alpha}^\top \mathbf{K} \boldsymbol{\alpha}, $$
> где $f_i = (\mathbf{K}\boldsymbol{\alpha})_i$ и $\lambda$ — параметр регуляризации. Переходя к переменным $\mathbf{f} = \mathbf{K}\boldsymbol{\alpha}$, получим эквивалентную задачу в пространстве значений:
> $$ \min_{\mathbf{f} \in \mathcal{R}(\mathbf{K})} \left\{ -\log p(\mathbf{y} \mid \mathbf{f}) + \frac{\lambda}{2} \mathbf{f}^\top \mathbf{K}^{-1} \mathbf{f} \right\}. $$
>
> При $\lambda = 1$ это в точности совпадает с задачей максимизации $\Psi(\mathbf{f})$ из (5.2). Следовательно, **MAP-оценка $\mathbf{f}_{\text{MAP}}$ в GP-классификации с нулевым средним есть решение KLR с единичным параметром регуляризации**. Более общо, если ввести шкалирование априорной ковариации (например, $\mathbf{f} \sim \mathcal{N}(\mathbf{0}, \sigma_f^2 \mathbf{K})$), то MAP-решение эквивалентно KLR с $\lambda = 1/\sigma_f^2$.
>
> Однако GP-классификация идёт дальше KLR: лапласовское приближение не только находит моду, но и строит гауссовскую аппроксимацию апостериора, давая ковариационную матрицу $\mathbf{\Sigma}_{\text{post}}$ и предсказательную дисперсию $\sigma_*^2$. KLR, будучи точечным методом, не предоставляет подобной меры неопределённости. Таким образом, лапласовское приближение можно рассматривать как *байесовское обобщение KLR*, позволяющее получать вероятностные прогнозы с учётом апостериорной неопределённости.
>
> Связь между итерациями Ньютона для MAP и IRLS в KLR также прозрачна: ньютоновский шаг (5.4) совпадает с шагом IRLS для KLR, если в последнем заменить регуляризацию $\lambda$ на единицу и соответствующим образом выбрать веса. Это подчёркивает, что лапласовское приближение унаследовало всю вычислительную эффективность IRLS, обогатив её байесовской интерпретацией.
>
> На практике лапласовское приближение хорошо работает, когда апостериор унимодален и близок к гауссовскому, но может давать смещённые оценки неопределённости для сильно нелинейных задач или малых выборок. В таких случаях применяют более точные, но и более затратные методы: MCMC, распространение ожидания (expectation propagation) или вариационный вывод.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Почему апостериорное распределение скрытых переменных $\mathbf{f}$ в GP-классификации не является гауссовским, в отличие от GP-регрессии?
2. Что такое лапласовское приближение? Опишите его идею: как оно строит гауссовскую аппроксимацию и какие величины для этого нужны?
3. Как найти MAP-оценку $\mathbf{f}_{\text{MAP}}$ в GP-классификации? Запишите уравнение (5.3) и объясните, почему оно нелинейно.
4. Как получить вероятность принадлежности классу $+1$ для новой точки $\mathbf{x}_*$ после лапласовского приближения? Какие шаги для этого выполняются?
5. В чём отличие GP-классификации с лапласовским приближением от ядерной логистической регрессии (KLR)? Даёт ли GP-классификация дополнительную информацию?

**Для профессионалов:**

1. Выведите гессиан $\nabla^2_{\mathbf{f}} \Psi$ логарифма апостериора и докажите, что матрица $\mathbf{K}^{-1} + \mathbf{W}$ положительно определена. Как это гарантирует единственность MAP-решения?
2. Докажите, что формула (5.8) для предсказательной дисперсии $f_*$ эквивалентна $\sigma_*^2 = k_{**} - \mathbf{k}_*^\top (\mathbf{K} + \mathbf{W}^{-1})^{-1} \mathbf{k}_*$, используя выражение (5.6) и свойства обратных матриц.
3. Сравните точность лапласовского приближения с оценками, полученными с помощью MCMC (например, Нётеровский гамильтонов Монте-Карло). В каких случаях лапласовское приближение даёт существенное смещение?
4. Как можно обобщить лапласовское приближение на многоклассовую классификацию (softmax-функция связи)? Опишите изменения в модели и структуре матрицы $\mathbf{W}$.
5. Выведите аппроксимацию интеграла $\int \sigma(f) \mathcal{N}(f \mid \mu, \sigma^2) df$ через функцию probit (или предложите другой аналитический приём). Почему эта аппроксимация удобна на практике?

### 6. Масштабирование гауссовских процессов: разреженные методы и индуктивные точки

Гауссовские процессы — мощный и элегантный вероятностный инструмент, однако их практическое применение долгое время сдерживалось высокой вычислительной сложностью. Как мы видели, обучение стандартного GP требует $O(n^3)$ операций для разложения Холецкого и $O(n^2)$ памяти для хранения матрицы $\mathbf{K}$. Это означает, что при $n \gtrsim 10^4$ прямой GP становится практически неприменимым на обычном оборудовании. К счастью, за последние два десятилетия разработан ряд методов, позволяющих масштабировать GP на сотни тысяч и миллионы точек, сохраняя при этом байесовскую интерпретируемость и оценки неопределённости. Центральное место среди них занимают **разреженные гауссовские процессы** (Sparse GPs), основанные на идее индуктивных точек и вариационном выводе.

---

#### 6.1. Проблема масштабирования и общая стратегия

Ключевые вычислительные узкие места полного GP:

- **Обучение:** $O(n^3)$ для разложения Холецкого или обращения матрицы $\mathbf{K}_y = \mathbf{K} + \sigma_n^2 \mathbf{I}$.
- **Память:** $O(n^2)$ для хранения $\mathbf{K}_y$.
- **Предсказание:** $O(n)$ на тестовую точку для среднего и $O(n^2)$ для дисперсии (если не использовать кэширование).

Общая идея масштабирования состоит в том, чтобы заменить полную ядерную матрицу её низкоранговым приближением. Вместо того чтобы работать со всеми $n$ обучающими точками, мы выбираем (или оптимизируем) небольшое множество из $m \ll n$ **индуктивных точек** (inducing points), которые служат «опорными» элементами, эффективно суммирующими информацию от всей выборки. Этот подход известен как **разреженный GP** (Sparse GP), а его современная формулировка через вариационный вывод (Titsias, 2009) обеспечивает не только вычислительную эффективность, но и строгие гарантии качества аппроксимации.

---

#### 6.2. Индуктивные точки и аппроксимация Найстрёма

Пусть $\mathbf{Z} = \{\mathbf{z}_1, \dots, \mathbf{z}_m\}$ — множество индуктивных точек в $\mathcal{X}$, не обязательно совпадающих с обучающими. Обозначим соответствующие значения скрытой функции как $\mathbf{u} = (f(\mathbf{z}_1), \dots, f(\mathbf{z}_m))^\top$. Априорно $\mathbf{u} \sim \mathcal{N}(\mathbf{0}, \mathbf{K}_{ZZ})$, где $\mathbf{K}_{ZZ}$ — матрица Грама для $\mathbf{Z}$. Ключевое предположение большинства разреженных методов состоит в том, что $\mathbf{u}$ содержит всю информацию, необходимую для предсказания в любой точке, а значения $\mathbf{f}$ на обучающих точках условно независимы при заданном $\mathbf{u}$ (или аппроксимируются как таковые). Это приводит к низкоранговой аппроксимации ядерной матрицы.

**Аппроксимация Найстрёма** — простейший способ получить такую аппроксимацию. Она основана на представлении

$$
\mathbf{K} \approx \mathbf{K}_{XZ} \mathbf{K}_{ZZ}^{-1} \mathbf{K}_{XZ}^\top,
$$

где $\mathbf{K}_{XZ}$ — матрица перекрёстных ядер между обучающими и индуктивными точками ($n \times m$). Соответственно, для предсказаний используется приближённое ядро

$$
\tilde{k}(\mathbf{x}, \mathbf{x}') = k(\mathbf{x}, \mathbf{Z}) \, \mathbf{K}_{ZZ}^{-1} \, k(\mathbf{Z}, \mathbf{x}').
$$

Такая аппроксимация имеет ранг не выше $m$, что снижает сложность обучения до $O(n m^2 + m^3)$ и память до $O(n m)$. В контексте GP её можно интерпретировать как параметрический метод: мы работаем с $m$-мерным вектором $\mathbf{u}$, и все вычисления сводятся к операциям с матрицами размера $m \times m$. Этот подход тесно связан с ядерной гребневой регрессией с Nyström-приближением (раздел 5.5), но в GP он дополнительно даёт меру неопределённости.

---

#### 6.3. Вариационный разреженный GP (Titsias, 2009)

Простой Nyström-подход является эвристическим и не гарантирует, что аппроксимированное апостериорное распределение близко к истинному. Революционный шаг сделал Титсиас (2009), предложив вариационную формулировку разреженного GP, которая одновременно оптимизирует и гиперпараметры, и расположение индуктивных точек, максимизируя нижнюю границу маргинального правдоподобия (ELBO).

В этой модели мы сохраняем совместное распределение всех латентных переменных:

$$
p(\mathbf{y}, \mathbf{f}, \mathbf{u}) = p(\mathbf{y} \mid \mathbf{f}) \, p(\mathbf{f} \mid \mathbf{u}) \, p(\mathbf{u}),
$$

где $p(\mathbf{f} \mid \mathbf{u})$ — условное распределение GP, а $p(\mathbf{u}) = \mathcal{N}(\mathbf{0}, \mathbf{K}_{ZZ})$. Истинный апостериор $p(\mathbf{f}, \mathbf{u} \mid \mathbf{y})$ аппроксимируется вариационным распределением вида

$$
q(\mathbf{f}, \mathbf{u}) = p(\mathbf{f} \mid \mathbf{u}) \, q(\mathbf{u}),
$$

где $q(\mathbf{u}) = \mathcal{N}(\mathbf{m}, \mathbf{S})$ — вариационное гауссовское распределение над $\mathbf{u}$, параметризуемое вектором средних $\mathbf{m}$ и ковариационной матрицей $\mathbf{S}$ (обычно полной или в форме Холецкого). Подставляя это в вариационный принцип, мы максимизируем нижнюю границу логарифмического маргинального правдоподобия:

$$
\log p(\mathbf{y}) \ge \mathbb{E}_{q(\mathbf{f}, \mathbf{u})}\!\left[ \log \frac{p(\mathbf{y} \mid \mathbf{f}) p(\mathbf{f} \mid \mathbf{u}) p(\mathbf{u})}{q(\mathbf{f}, \mathbf{u})} \right] = \mathbb{E}_{q(\mathbf{f})}[\log p(\mathbf{y} \mid \mathbf{f})] - \text{KL}(q(\mathbf{u}) \| p(\mathbf{u})).
$$

Благодаря структуре $q(\mathbf{f}) = \int p(\mathbf{f} \mid \mathbf{u}) q(\mathbf{u}) d\mathbf{u}$, маргинальное распределение $\mathbf{f}$ также гауссовское:

$$
\mathbf{f} \sim \mathcal{N}\bigl( \mathbf{K}_{XZ} \mathbf{K}_{ZZ}^{-1} \mathbf{m}, \; \mathbf{K}_{XX} - \mathbf{K}_{XZ} \mathbf{K}_{ZZ}^{-1} (\mathbf{K}_{ZZ} - \mathbf{S}) \mathbf{K}_{ZZ}^{-1} \mathbf{K}_{XZ}^\top \bigr).
$$

Ожидаемое логарифмическое правдоподобие $\mathbb{E}_{q(\mathbf{f})}[\log p(\mathbf{y} \mid \mathbf{f})]$ для регрессии с гауссовским шумом вычисляется аналитически и даёт разреженную версию GP, а для классификации требует дополнительной аппроксимации (например, через квадратуру). KL-дивергенция между $q(\mathbf{u})$ и $p(\mathbf{u})$ имеет замкнутую форму как расстояние между двумя нормальными распределениями.

Оптимизация проводится по трём группам параметров:

- **Вариационные параметры** $\mathbf{m}$ и $\mathbf{S}$ (или их параметризация);
- **Индуктивные точки** $\mathbf{Z}$ (рассматриваются как свободные параметры и оптимизируются градиентным спуском вместе с остальными);
- **Гиперпараметры ядра** и шума.

Сложность каждой итерации составляет $O(n m^2 + m^3)$, что при $m \approx 100 \dots 500$ делает метод применимым к $n \gtrsim 10^5$. Более того, вариационная природа даёт гарантии: мы максимизируем нижнюю границу, поэтому увеличение $m$ всегда может улучшить аппроксимацию.

---

#### 6.4. Использование GPyTorch для разреженного GP (кратко)

Библиотека GPyTorch (на базе PyTorch) реализует описанный вариационный метод в рамках модульной архитектуры. Создание разреженного GP для $n=10^4$ точек с $m=100$ индуктивными точками выглядит следующим образом (псевдокод):

- Определяем модель: `VariationalGPModel` с ядром (например, RBF) и `InducingPoints` с начальным расположением $\mathbf{Z}$, выбранным, скажем, через k-means на подмножестве $\mathbf{X}$.
- Используем `VariationalELBO` как критерий оптимизации.
- Обучаем с помощью Adam или L-BFGS, совместно настраивая гиперпараметры и положения $\mathbf{Z}$.

Эксперимент на синтетических данных (например, тот же зашумлённый синус, но с $n=10^4$) показывает:

- Время обучения полного GP — порядка нескольких минут и кубический рост.
- Время обучения разреженного GP — секунды, рост линейный по $n$.
- Предсказания среднего и доверительные полосы практически неотличимы от полного GP при достаточно большом $m$ (обычно 100–200).
- При $m \approx 50$ возможна небольшая потеря точности, особенно на экстраполяции.

---

#### 6.5. Другие приближения и современные методы

Помимо вариационного разреженного GP, существуют и другие подходы к масштабированию:

- **Случайные признаки Фурье (Random Fourier Features, RFF).** Для стационарных ядер (теорема Бохнера) генерируются $D$ случайных гармоник, и GP сводится к байесовской линейной регрессии в $D$-мерном пространстве. Сложность $O(n D^2 + D^3)$, подходит для экстремально больших $n$.
- **Структурированные ядра (KISS-GP).** Если обучающие точки лежат на регулярной сетке, умножение на $\mathbf{K}$ можно ускорить через быстрое преобразование Фурье, снижая сложность до $O(n \log n)$. Метод Кронекеровых и Тёплицевых структур (Kronecker, Toeplitz) распространяет эту идею на многомерные сетки.
- **Локальные GP.** Разделение пространства на подобласти и обучение независимых (или слабозависимых) GP в каждой из них. Прост в реализации, но страдает на стыках областей.
- **Использование GPU и параллельных вычислений.** Библиотеки вроде GPyTorch используют эффективные матрично-матричные умножения на GPU (BlackBox Matrix-Matrix Multiplication, BBMM) и метод сопряжённых градиентов, позволяя обучать GP с $n$ до нескольких миллионов.

Каждый из этих методов вносит свой компромисс между точностью, вычислительной сложностью и удобством реализации. Вариационный разреженный GP остаётся золотым стандартом благодаря своей строгой вероятностной основе, возможности обучения индуктивных точек и хорошей эмпирической производительности.

---

#### Вопросы для самопроверки (по всей теме GP)

**Для начинающих:**

1. В чём заключается проблема $O(n^3)$ и при каком размере выборки стандартный GP становится непрактичным?
2. Что такое индуктивные точки? Какую роль они играют в разреженном GP?
3. Как работает аппроксимация Найстрёма для ядерной матрицы? Напишите формулу приближённого ядра.
4. Что такое ELBO и зачем её максимизируют в вариационном разреженном GP?
5. Зачем нужно разреженное приближение, если можно использовать полный GP? В каких ситуациях разреженный GP предпочтительнее?
6. Назовите два других метода масштабирования GP, помимо разреженных.

**Для профессионалов:**

1. Выведите вариационную нижнюю границу (ELBO) для разреженного GP (Titsias, 2009) в случае регрессии. Покажите, как она распадается на сумму ожидаемого правдоподобия и KL-дивергенции.
2. Объясните, как выбрать расположение индуктивных точек $\mathbf{Z}$. Сравните подходы: фиксированные на сетке, k-means на $\mathbf{X}$, оптимизация вместе с параметрами.
3. Сравните точность sparse GP и полного GP на синтетических данных (например, функция Матерна). Как зависит ошибка от числа индуктивных точек $m$?
4. Реализуйте разреженный GP с использованием только `scipy` (без GPyTorch): постройте Nyström-аппроксимацию, вычислите апостериор и предсказания. Сравните время с полным GP.
5. Обсудите связь между разреженным GP и KRR с Nyström-приближением. В каком смысле sparse GP является байесовским обобщением Nyström-KRR? Какие дополнительные преимущества даёт вариационная формулировка?